This project is based on the use of Steam’s dataset relating to its platform. Steam is the most popular online gaming platform on the market, with over 50,000 games available on its platform in the year this dataset was published. 

We will therefore carry out a descriptive analysis simulation for Ubisoft, in order to determine the typical profile of games we would recommend they produce.

During this analysis, we will answer the following questions: 

#### Analysis at the "macro" level:

- Which publisher has released the most games on Steam?
- What are the best rated games?
- Are there years with more releases? Were there more or fewer game releases during the Covid, for example?
- How are the prizes distributed? Are there many games with a discount?
- What are the most represented languages?
- Are there many games prohibited for children under 16/18?

#### Genres analysis:

- What are the most represented genres?
- Are there any genres that have a better positive/negative review ratio?
- Do some publishers have favorite genres?
- What are the most lucrative genres?

#### Platform analysis:

- Are most games available on Windows/Mac/Linux instead?
- Do certain genres tend to be preferentially available on certain platforms?


# 1. Importation and overview 

First, we will import the data rom the S3 Bucket and take a general look at the dataset. 

In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F 
from pyspark.sql.types import DataType
from pyspark.sql.types import IntegerType


In [0]:
# Creation of the Spark session
spark = SparkSession.builder \
    .appName("Steam") \
    .getOrCreate() 

# Importation of the Steam Game Dataset
df = spark.read.json("s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json")

Let’s look at how the information is organised 

In [0]:
# Let's take a look at the overall structure of the dataset
df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

First, we will load the data into the memory. This will allow subsequent operations to run faster. 

In [0]:
df.count()

55691

We can already notice that the dates are stored in strings. We will therefore need to convert them before creating new tables for our analyses. 

# 2. Macro analysis 

First, we will answere general analysis questions about the dataset : 

## 2.1. Best publisher 

So, the first question is :     
Which publisher has released the most games on Steam?

In [0]:
display(df.select("data.publisher", "data.name").limit(10))


publisher,name
Valve,Counter-Strike
PsychoFlux Entertainment,ASCENXION
"Team17, NEXT Studios",Crown Trick
Vertigo Gaming Inc.,"Cook, Serve, Delicious! 3?!"
DoubleC Games,细胞战争
2P Games,Zengeon
Starship Studio,干支セトラ 陽ノ卷｜干支etc. 陽之卷
重庆环游者网络科技,Jumping Master(跳跳大咖)
Simon Codrington,Cube Defender
Villain Role,Tower of Origin2-Worm's Nest


We have the publisher names and the names of all publishers associated with the games they have released. 

In [0]:
display(
    df.select("data.publisher", "data.name")
    .filter(F.col("data.publisher") == "Valve")
    .limit(5)
)

publisher,name
Valve,Counter-Strike
Valve,Dota Underlords
Valve,Artifact Foundry
Valve,Half-Life: Blue Shift
Valve,Source Filmmaker


And each game correctly corresponds to a single row with its associated publisher. We will therefore sort them accordingly.

In [0]:
# We create a temporary view to allow faster filtering using SQL
df.createOrReplaceTempView("df_sql")

In [0]:
display(spark.sql(f"""
    SELECT data.publisher, COUNT(data.name) as number_games
    FROM df_sql
    GROUP BY data.publisher
    ORDER BY number_games DESC
    LIMIT 5
"""))

publisher,number_games
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141


So, the top 3 publishers with the most games released on Steam are:   
- Big Fish Games with 422 games
- 8florr with 222 games 
- SEGA with 165 games 

We can note that Big Fish Games is far ahead, with almost twice as many games as 8floor, which is in second place.

## 2.2. Best game 

We will now look at which games received the highest ratings.

In [0]:
display(
  df.select("data.name", "data.positive")
  .orderBy(F.desc("data.positive"))
  .limit(10)
)

name,positive
Counter-Strike: Global Offensive,5943345
Dota 2,1534895
Grand Theft Auto V,1229265
PUBG: BATTLEGROUNDS,1185361
Terraria,1014711
Tom Clancy's Rainbow Six Siege,942910
Garry's Mod,861240
Team Fortress 2,846407
Rust,732513
Left 4 Dead 2,643836


The highest-rated game is Counter-Strike: Global Offensive, and by a large margin: it is rated about three times higher than Dota 2, which is in second place. It is followed by Grand Theft Auto V and PUBG: BATTLEGROUNDS, which have fairly similar scores.

## 2.3. Game releases by year

We will now look at whether there were years with a higher number of game releases.   
We will then focus on the impact of COVID-19 on game releases.

In [0]:
display(df.select("data.release_date").limit(5))
print(df.select("data.release_date").dtypes)

release_date
2000/11/1
2021/05/14
2020/10/16
2020/10/14
2019/03/30


[('release_date', 'string')]


The dates are stored as strings in the y/m/d format.    
We will therefore need to use a TimeStamp to convert them. 

In [0]:
# In this version of Spark, try_to_timestamp is not available in Python
# We therefore need to use SQL through F.expr instead.
date = df.withColumn("date_clean",
    F.expr("try_to_timestamp(data.release_date, 'yyyy/M/d')")
)

display(date.select("date_clean").limit(5))


date_clean
2000-11-01T00:00:00.000Z
2021-05-14T00:00:00.000Z
2020-10-16T00:00:00.000Z
2020-10-14T00:00:00.000Z
2019-03-30T00:00:00.000Z


In [0]:
display(
    date.groupBy(F.year("date_clean").alias("year"))
    .agg(F.count("data.name").alias("number_games"))
    .orderBy(F.desc("number_games"))
)

year,number_games
2021,8805
2020,8287
2018,7663
2022,7451
2019,6949
2017,6006
2016,4176
2015,2566
2014,1550
2013,469


The peak in game releases on Steam corresponds to the lockdown years due to COVID, in 2020 and 2021 with over 8 000 games released.    
We observe a clear increase since around 2015, which corresponds to the period when fiber deployment significantly accelerated worldwide.

## 2.4. Price distribution 



In [0]:
display(df.select("data.price").limit(5))
print(df.select("data.price").dtypes)

price
999
999
599
1999
199


[('price', 'string')]


Prices are in cents, so they need to be converted. They are also stored as strings, but Spark automatically handles the conversion when dividing by a number.

In [0]:
price = df.withColumn("prince_clean", F.col("data.price") / 100)
price.show(5)

+--------------------+-------+------------+
|                data|     id|prince_clean|
+--------------------+-------+------------+
|{10, [Multi-playe...|     10|        9.99|
|{1000000, [Single...|1000000|        9.99|
|{1000010, [Single...|1000010|        5.99|
|{1000030, [Multi-...|1000030|       19.99|
|{1000040, [Single...|1000040|        1.99|
+--------------------+-------+------------+
only showing top 5 rows


The subfields are nested within the data, so they cannot be easily manipulated. We will therefore create a flattened version of the dataset to work with them more freely.

In [0]:
df_data = df.select("data.*")

price = (df_data
         # We filter only the games
         # and convert the cents
         .filter(F.col("type") == "game")
         .withColumn("price_clean", F.col("price") / 100)
)
display(price.select("price_clean").limit(20))

price_clean
9.99
9.99
5.99
19.99
1.99
7.99
12.99
0.0
2.99
13.99


In [0]:
display(price.select("price_clean").describe())

summary,price_clean
count,55690
mean,7.732988687371752
stddev,10.931394858517274
min,0.0
max,999.0


We get a strange result with 999, which is no longer in cents. Let’s look at the number of anomalous values.

In [0]:
display(
    price.filter(F.col("price_clean") > 90)
    .select("price")
    .describe()
)

summary,price
count,66
mean,15636.80303030303
stddev,11678.613567898305
min,11999
max,9999


We note that there are 66 outliers out of the 55,691 values in the dataset.
Also, we need to make sure that we are only analyzing games and not other types of content available on Steam.

In [0]:
price_range = price.withColumn("price_range",
        # We add a prefix to ensure the data appears in the correct order in the visualization.
        F.when(F.col("price_clean") == 0, "a: 0") 
        .when(F.col("price_clean") < 5, "b: 0-5")
        .when(F.col("price_clean") < 10, "c: 5-10")
        .when(F.col("price_clean") < 20, "d: 10-20")
        .when(F.col("price_clean") < 50, "e: 20-50")
        .otherwise("f: 50+")
)

display(
    price_range
    .groupBy("price_range")
    .agg(F.count("*").alias("number_game"))
    .orderBy("price_range")
)

price_range,number_game
a: 0,7779
b: 0-5,23478
c: 5-10,12450
d: 10-20,9022
e: 20-50,2633
f: 50+,328


Databricks visualization. Run in Databricks to view.

The majority of games sold on Steam are inexpensive: they are priced between $0 and $10. The price range that stands out the most is the $0 to $5 category.   
Few games sell for more than $20 and games sold for more than $50 are a minority.
  
Let’s now take a look at discounted games.

In [0]:
display(df_data.select('discount', 'name').limit(5))

discount,name
0,Counter-Strike
0,ASCENXION
70,Crown Trick
0,"Cook, Serve, Delicious! 3?!"
0,细胞战争


We can observe that the discount column contains the percentage of reduction for each game.

In [0]:
# Let’s look at the discount rate.
discount_true = df_data.filter(F.col('discount') != 0)
display(discount_true.select('discount').describe()) # Without using select, describe takes all columns.

# We’ll take this opportunity to count the total number of games in the dataset
total = df_data.filter(F.col("type") == "game").count()

# We have already calculat the total of game
discount_pourcentage = (discount_true.count() / total) * 100
print(f" there is {discount_pourcentage:.2f}% of game on discount on Steam")

summary,discount
count,2518
mean,57.58816521048451
stddev,22.512945353983145
min,10
max,90


 there is 4.52% of game on discount on Steam


In [0]:
# Let’s also look at the discount rate by price range.
agg_expr =(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("discount") != "0", 1).otherwise(0)).alias("games_with_discount")
)

print("Discount percentage by price range")
display(
    price_range
    .groupBy("price_range")
    .agg(*agg_expr)
    .withColumn("discount_percentage", 
        F.round(F.col("games_with_discount") / F.col("total") * 100, 2)
    )
    .orderBy("price_range")
)

Discount percentage by price range


price_range,total,games_with_discount,discount_percentage
a: 0,7779,0,0.0
b: 0-5,23478,1884,8.02
c: 5-10,12450,380,3.05
d: 10-20,9022,211,2.34
e: 20-50,2633,41,1.56
f: 50+,328,2,0.61


We observe that with only 4.5% of games on discount, there are very few promotions on Steam. Games are already relatively inexpensive. What is interesting is that the cheapest games are the ones that apply discounts by far the most. These discounts are also quite significant, with an average of 57%.

## 2.5. Language

We will now look at which languages are the most represented.

In [0]:
display(df_data.select("languages").limit(10))

languages
"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean"
"English, Korean, Simplified Chinese"
"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil"
English
Simplified Chinese
"Simplified Chinese, English, Traditional Chinese, Japanese, Korean"
"Japanese, Simplified Chinese, Traditional Chinese"
"English, Simplified Chinese, Traditional Chinese"
English
"English, Simplified Chinese, Traditional Chinese"


In [0]:
df_languages = (df_data.withColumn("language_in_all",
                                   F.explode(F.split(F.col("languages"), ", ")
                                   ))
)

display(
    df_languages
    .groupBy("language_in_all")
    .agg(F.count("language_in_all").alias("total_number_languages"))
    .orderBy(F.desc("total_number_languages"))
    .limit(20)
)

language_in_all,total_number_languages
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6599


Databricks visualization. Run in Databricks to view.

English is present in almost all games.
It is followed by German, French, Russian, Chinese, and Spanish, which each appear in roughly 20% of games.

## 2.6. Age restriction 

We are going to look at what proportion of games are restricted to those under 16 and 18.

In [0]:
display(df_data.select('required_age').limit(10))
print(df_data.select('required_age').describe())


required_age
0
0
0
0
0
0
0
0
0
0


DataFrame[summary: string, required_age: string]


To filter this column, you therefore need to convert the strings to integers

To do this, we’ll use the `regexp_extract` function:
https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_extract.html 

In [0]:
df_data = df_data.withColumn("required_age_int", 
                             F.regexp_extract(F.col("required_age"), r"(\d+)", 1).cast("int"))


display(
    df_data.groupBy("required_age_int")
    .agg(F.count("*").alias("number_games"))
    .withColumn("percentage",
                F.round(F.col("number_games") / total * 100, 2))
    .orderBy("required_age_int")
)

required_age_int,number_games,percentage
0,55030,98.81
3,3,0.01
5,1,0.0
6,4,0.01
7,3,0.01
8,3,0.01
9,1,0.0
10,7,0.01
12,32,0.06
13,26,0.05


There are some outliers, such as age limits of 35 and 180. These should therefore be removed. 

In [0]:
df_data = df_data.filter(F.col('required_age_int') <= 21)

Now we can filter out everyone over the age of 16 and 18.

In [0]:
age_16_min = df_data.filter(F.col("required_age_int") >= 16).count()
age_18_min = df_data.filter(F.col("required_age_int") >= 18).count()

age_16_min_percentage = round(age_16_min / total * 100, 2)
age_18_min_percentage = round(age_18_min / total * 100, 2)

print(f"The percentage of games restricted to those aged 16 and over is:{age_16_min_percentage}%")
print(f"The percentage of games restricted to those aged 16 and over is:{age_18_min_percentage}%")

The percentage of games restricted to those aged 16 and over is:0.54%
The percentage of games restricted to those aged 16 and over is:0.4%


The percentage of games restricted to those aged 16 or 18 and over is 0.54% and 0.4% respectively. Furthermore, 98.81% of games have no age restrictions.     
Steam is therefore a platform on which almost all games are suitable for all ages. 

## 2.7. Conclusion of the macro analysis 

There are clear trends that characterise the most popular games on Steam.
These are mainstream titles, and most of them have no age restriction. They can also be easily purchased or downloaded by young people, as their price generally ranges from zero to twenty dollars, with the majority costing between zero and ten dollars. Finally, almost all of them are in English, and most offer German, French, Russian, Chinese and Spanish as optional languages. 

Now we’re going to take a closer look at the types of games that are most popular 

# 3. Genre analysis 
## 3.1. The most common genres

First of all, we’re going to look at which genres are most prevalent on Steam

In [0]:
display(df_data.select("genre").limit(20))
display(df_data.select("genre").distinct())

genre
Action
"Action, Adventure, Indie"
"Adventure, Indie, RPG, Strategy"
"Action, Indie, Simulation, Strategy"
"Action, Casual, Indie, Simulation"
"Action, Adventure, Indie, RPG"
"Adventure, Indie, RPG, Strategy"
"Action, Adventure, Casual, Free to Play, Massively Multiplayer"
"Casual, Indie"
"Indie, RPG"


genre
"Indie, RPG"
"RPG, Strategy, Early Access"
"Adventure, RPG, Strategy"
"Action, Casual, Indie, RPG, Sports"
"Adventure, Casual, Indie, RPG, Simulation, Early Access"
"Adventure, Casual, Indie, RPG, Strategy, Early Access"
"Action, Indie, RPG"
"Casual, Indie, Simulation, Early Access"
"Action, Free to Play, Indie, Early Access"
"Action, Adventure, Casual, Indie, Racing, Simulation"


In [0]:
df_genres_split = (df_data
                  .filter(F.col("type") == "game")
                  .withColumn("genre_split", F.explode(F.split(F.col("genre"), ", |& | &")))
)

df_genres = (df_genres_split
        .groupBy("genre_split")
        .agg(F.count("*").alias("number_software"))
        .withColumn("percentage", F.round(F.col("number_software") / total * 100, 2))
        .orderBy(F.desc("number_software"))
)
display(df_genres)

genre_split,number_software,percentage
Indie,39679,71.25
Action,23757,42.66
Casual,22086,39.66
Adventure,21431,38.48
Strategy,10895,19.56
Simulation,10831,19.45
RPG,9533,17.12
Early Access,6143,11.03
Free to Play,3393,6.09
Sports,2665,4.79


It is clear that the overwhelming majority of software available on Steam consists of indie games, accounting for 71% of releases. It is therefore mainly small studios with limited budgets and small teams (sometimes consisting of just one person) that dominate the platform. This is reflected in the price of most games, which ranged from zero to twenty euros. 

In second place, with 43%, are action games, such as Counter-Strike: Global Offensive, which was the best-selling title on the platform. 

In third place are casual games and adventure games, with 39% and 38% respectively. These two types are diametrically opposed to one another: casual games are low-commitment, meaning they have simple rules and are generally played in a matter of minutes. In contrast, adventure games require a long-term commitment and often feature complex gameplay mechanics. 

As for the least common genres, we find so-called violent games at 0.3%, racing games at 4% and sports games at 5%. 

## 3.2. Reviews by genre 

We have noted the presence of these genres on the platform; we will now look at their popularity 

In [0]:
# Let’s take a look at what the Positiv column looks like
display(df_data.select("positive").limit(10))

positive
201215
27
4032
1575
0
1018
18
50
6
32


In [0]:
total_positive_reviews = df_genres_split.agg(F.sum("positive")).first()[0]

display(df_genres_split
                   .groupBy("genre_split")
                   .agg(F.sum("positive").alias("total_positive"))
                   .withColumn("positive_percentage", F.round(F.col("total_positive")/ total_positive_reviews * 100))
                   .orderBy(F.desc("total_positive"))
)

genre_split,total_positive,positive_percentage
Action,54858394,25.0
Indie,32530835,15.0
Adventure,29689445,14.0
RPG,19425366,9.0
Free to Play,18722246,9.0
Simulation,15570849,7.0
Strategy,13402870,6.0
Casual,10034967,5.0
Massively Multiplayer,7979078,4.0
Early Access,4334431,2.0


The ‘Action’ genre stands out clearly from the rest, with 25% of positive reviews. It is followed by the indie genre with 15% and adventure with 14%. These are the most common genres, with the exception of the casual genre, which accounts for only 5%. 


In [0]:
total_negative_reviews = df_genres_split.agg(F.sum("negative")).first()[0]

display(df_genres_split
                   .groupBy("genre_split")
                   .agg(F.sum("negative").alias("total_negative"))
                   .withColumn("negative_percentage", F.round(F.col("total_negative")/ total_negative_reviews * 100))
                   .orderBy(F.desc("total_negative"))
)

genre_split,total_negative,negative_percentage
Action,9687391,25.0
Adventure,5653153,15.0
Free to Play,4280807,11.0
Indie,4240959,11.0
RPG,3274085,8.0
Massively Multiplayer,2938660,8.0
Simulation,2400011,6.0
Strategy,2393425,6.0
Casual,1537296,4.0
Early Access,935935,2.0


We see exactly the same distribution in negative reviews as in positive reviews. This means that the reviews give us an accurate picture of the popularity of Steam games. This is interesting information because our dataset does not contain any data on sales figures. 

There are a few interesting points to note regarding the ratio of negative to positive reviews: 
Free-to-play games rank third in negative reviews rather than fifth in positive reviews. This is because players who pay for a game first select the games that best suit them. 
Conversely, ‘Indie’ accounts for only 11% of negative reviews, whereas it accounts for 15% of positive reviews. Players therefore rate games more highly when they have a genuine artistic vision and offer original content. 

Translated with DeepL.com (free version)


Let’s now take a look at the genres that the biggest and biggest publishers tend to favour 

## 3.3. Genre by publisher 

In [0]:
# Top 3 publishers
top_publishers = (df_data
    .filter(F.col("type") == "game")
    .filter(F.col("publisher").isNotNull())
    .filter(F.col("publisher") != "")
    .groupBy("publisher")
    .agg(F.count("*").alias("number_games"))
    .orderBy(F.desc("number_games"))
    .limit(5)
)

# favourite genre
display(
    df_genres_split
    .filter(F.col("publisher").isin([r["publisher"] for r in top_publishers.collect()]))
    .groupBy("publisher", "genre_split")
    .agg(F.count("*").alias("number_games"))
    .orderBy("publisher", F.desc("number_games"))
)

publisher,genre_split,number_games
8floor,Casual,202
8floor,Strategy,22
8floor,Simulation,10
8floor,Adventure,8
8floor,Indie,1
Big Fish Games,Casual,418
Big Fish Games,Adventure,392
Big Fish Games,Indie,7
Big Fish Games,Simulation,7
Big Fish Games,Strategy,6


The most prominent publisher, Big Fish Games, has roughly half of its games in the casual genre. This aligns with the nature of this type of game, which requires little time to complete. It also develops a significant number of adventure games. Its strategy, therefore, is to cover both games that require a high level of engagement and those that require a low level of engagement. 

The second-largest publisher, 8floor, also has a majority of casual games. This explains why so many of its games are published on Steam. 

As for SEGA, we can see a large proportion of action games, followed by adventure and strategy games. It has a very small share of casual games. Its strong presence is therefore due to its long history. It does not follow the example of the other two largest publishers, who flood Steam with short games. 

The same applies to Square Enix, which has also been a publisher since before the advent of game streaming. It also favours traditional game genres such as action and RPGs. 

These observations regarding these publishers can be explained by economic constraints. Traditional publishing companies, which already have financial resources, can directly fund substantial games in genres such as action, adventure and RPGs. 
Conversely, new publishers have adopted a strategy of releasing a large number of short games, known as casual games. This is intended to provide them with a financial foundation from which they can subsequently develop more elaborate games in genres such as adventure and strategy. 


We’ve looked at the most common types; now let’s look at the most profitable ones 

## 3.4. The most lucrative genres 

In [0]:
display(df_genres_split.select("price").limit(10))
print(df_genres_split.select("price").limit(10))

display(df_genres_split.select("owners").limit(10))
print(df_genres_split.select("owners").limit(10))


price
999
999
999
999
599
599
599
599
1999
1999


DataFrame[price: string]


owners
"10,000,000 .. 20,000,000"
"0 .. 20,000"
"0 .. 20,000"
"0 .. 20,000"
"200,000 .. 500,000"
"200,000 .. 500,000"
"200,000 .. 500,000"
"200,000 .. 500,000"
"100,000 .. 200,000"
"100,000 .. 200,000"


DataFrame[owners: string]


In [0]:
# First, we need to convert the string of owners, replacing the commas in the numbers with full stops
df_genres_split = df_genres_split.withColumn("owners_clean",
    F.regexp_replace(
        F.regexp_extract(F.col("owners"), r"^[\d,]+", 0),
        ",", "" 
    ).cast("long")
)

# and convert the price to a whole number and to dollars 
df_genres_split = df_genres_split.withColumn("price_clean", F.col("price").cast("double") / 100)


In [0]:
display(
    df_genres_split
    .groupBy("genre_split")
    .agg(F.sum(F.col("price_clean") * F.col("owners_clean")).alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)

genre_split,total_revenue
Action,3.59292303E10
Adventure,2.26189065E10
Indie,1.91346772E10
RPG,1.66750885E10
Strategy,1.23623921E10
Simulation,1.1422144E10
Casual,4.4765685E9
Massively Multiplayer,3.6927213E9
Early Access,3.147027E9
Sports,1.8160136E9


Databricks visualization. Run in Databricks to view.

Since Steam does not publish actual sales figures, we use the number of owners as an approximation. The figures are not exact, but we can still estimate which genres are the most lucrative. 

The action genre stands out from the rest as by far the most lucrative. This is consistent with the fact that it is also the most prevalent and most popular genre on the platform. 
Behind it come the adventure and indie genres. This corresponds to the ranking of the most popular games. 
More generally, we can see that the ranking of the most profitable genres broadly follows that of the most popular genres on the Steam platform. Casual, although the most prevalent genre, ranks only seventh. This is undoubtedly why publishers investing in it choose to produce them in large numbers in the hope of generating a reliable income from them. 
Finally, it is worth noting that the least profitable genres are those relating to sports and motor racing games


## 3.5. Conclusion of genres analysis 

The action genre clearly dominates the market on Steam. Its total number of reviews makes it the most popular and most lucrative genre. However, there is fierce competition in this sector, as it accounts for 43% of the most prevalent genres on Steam. 

Major publishers seem to be focusing on producing adventure and RPG games. Their presence on the platform is weaker than that of action games, even though they are among the most lucrative. A publisher offering this type of game therefore has a better chance of their games being noticed. 

It is interesting to note the popularity of the indie genre. At 71%, it is by far the most prevalent genre on the platform. It also has the best ratio of positive to negative reviews. This suggests that players reward games with a strong artistic vision and original gameplay mechanics. 

Finally, we observe that casual is a very distinctive genre. Although very prevalent on the platform, it attracts very few reviews, whether positive or negative. This is due to the nature of these games: quick-play titles whose sole purpose is simply to pass the time. Players use them but do not go so far as to post a review, which suggests that a certain depth is required in games to truly capture the public’s interest. Nevertheless, they are produced in large numbers by the biggest developers. They therefore enable certain studios to generate revenue, which in turn allows them to fund more complex games. 

Now let’s look at which platform is best for posting 

# 4. Operating systems
## 4.1. Presence of operating systems

Let’s take a look at the rankings on Steam 

In [0]:

total_games = df_data.filter(F.col("type") == "game").count()

display(
    df_data
    .filter(F.col("type") == "game")
    .agg(
        F.round(F.sum(F.col("platforms.windows").cast("int")) / total_games * 100, 2).alias("windows_pct"),
        F.round(F.sum(F.col("platforms.mac").cast("int")) / total_games * 100, 2).alias("mac_pct"),
        F.round(F.sum(F.col("platforms.linux").cast("int")) / total_games * 100, 2).alias("linux_pct")
    )
)

windows_pct,mac_pct,linux_pct
99.97,22.93,15.19


Almost all games on Steam run on Windows. 22% of games can also be played on a Mac, and 15% can also run on Linux. Windows is therefore the undisputed go-to operating system for releasing games. The percentages for Mac and Linux are low. The vast majority of players therefore appear to own a Windows PC or use an emulator. 

Let’s now look at the distribution of bones by genus

## 4.2. Genre by operating systems


In [0]:
# the most common genres on Mac 
display(df_genres_split
        .filter(F.col("genre_split").isNotNull())
        .filter(F.col("genre_split") != "")
        .groupBy("genre_split")
        .agg(
               F.round(F.avg(F.col("platforms.windows").cast("int")) * 100, 2).alias("windows_genres_pct"),
               F.round(F.avg(F.col("platforms.mac").cast("int")) * 100, 2).alias("mac_genres_pct"),
               F.round(F.avg(F.col("platforms.linux").cast("int")) * 100, 2).alias("linux_genres_pct")
               )
        .orderBy(F.desc("mac_genres_pct"))
)


genre_split,windows_genres_pct,mac_genres_pct,linux_genres_pct
Game Development,100.0,32.7,22.01
Strategy,99.97,27.58,16.76
Indie,99.99,25.04,17.59
Accounting,100.0,25.0,0.0
Free to Play,99.94,24.9,13.97
Design,99.75,24.63,13.3
Illustration,99.75,24.63,13.3
Sexual Content,100.0,24.07,12.96
RPG,99.99,23.58,15.99
Adventure,99.98,23.51,15.41


In [0]:
display(df_genres_split
        .filter(F.col("genre_split").isNotNull())
        .filter(F.col("genre_split") != "")
        .groupBy("genre_split")
        .agg(
               F.round(F.avg(F.col("platforms.windows").cast("int")) * 100, 2).alias("windows_genres_pct"),
               F.round(F.avg(F.col("platforms.mac").cast("int")) * 100, 2).alias("mac_genres_pct"),
               F.round(F.avg(F.col("platforms.linux").cast("int")) * 100, 2).alias("linux_genres_pct")
               )
        .orderBy(F.desc("linux_genres_pct"))
)

genre_split,windows_genres_pct,mac_genres_pct,linux_genres_pct
Game Development,100.0,32.7,22.01
Indie,99.99,25.04,17.59
Strategy,99.97,27.58,16.76
RPG,99.99,23.58,15.99
Nudity,100.0,22.22,15.56
Adventure,99.98,23.51,15.41
Casual,99.98,23.23,14.96
Action,99.98,19.21,14.22
Simulation,99.96,22.52,14.14
Gore,100.0,20.2,14.14



The data confirms Windows’ position as a universal operating system. 

As for Mac, it is present across all types of platforms. It is worth noting that Macs are used more frequently for creative software, such as design or illustration tools, accounting for 24%. It is also present in video, photo and programming software, with a significant share of around 15%. This corresponds to Apple’s product segmentation, which targets an audience of professional creatives. 

Linux, on the other hand, is used more for gaming and has very little presence in video or photo creation software.

# 5. Conclusion 

As for the general parameters, if Ubisoft wants to maximise its chances of becoming popular on Steam, it should release games in a price range of between $0 and $20. More specifically, the price should be around $5 to reach the widest possible audience. The games should also have no age restrictions. As for available languages, the games must be in English, and potentially include language options in German, French, Russian, Chinese and Spanish. The games must be for Windows. Given that the typical game would cost around $5, the studio would have a limited budget. It is therefore not necessary to port the games to Mac and Linux, unless the software used in production allows for automatic porting between operating systems, such as Unreal Engine or Unity. 

As for the type of game that would potentially be most successful, action is by far the most favoured genre. The adventure genre, meanwhile, is the one that garners the most positive feedback relative to its market presence. Developing an action-adventure game would be an excellent way to build player loyalty by maximising public recognition. Furthermore, Ubisoft’s biggest competitors would be indie games, which account for 71% of the games available on the platform. Products would therefore need to have a very distinctive artistic direction, with a preference for original gameplay mechanics. This would be a good approach to counter the competition from games by independent studios. 

Finally, it is worth noting that the current dataset does not contain information regarding game sales. Our analysis is therefore based on an estimate of the games’ popularity among players, using their ratio of positive to negative reviews, their presence on the platform, and the estimated revenue they generate. If we wish to take the analysis further, sales data would need to be included. 